# CNN on CIFAR-10

AIO 2026, Module 5, Lesson 1 (From MLP to CNN). This notebook builds the "simple CNN"
from the lecture (four 7×7 convolutions, no pooling, no padding) and compares it with
the MLP baseline from `CIFAR10_MLP.ipynb`.

## 1. Data

In [7]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
from torchsummary import summary

transform = transforms.ToTensor()

train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

## 2. Model

| Layer | Kernels | Output shape (C, H, W) | Params |
|---|---|---|---|
| Input | | (3, 32, 32) | |
| Conv1 + ReLU | 32 × (3, 7, 7) | (32, 26, 26) | 4,736 |
| Conv2 + ReLU | 64 × (32, 7, 7) | (64, 20, 20) | 100,416 |
| Conv3 + ReLU | 128 × (64, 7, 7) | (128, 14, 14) | 401,536 |
| Conv4 + ReLU | 256 × (128, 7, 7) | (256, 8, 8) | 1,605,888 |
| Flatten | | 16,384 | |
| Linear + ReLU | | 128 | 2,097,280 |
| Linear | | 10 | 1,290 |
| **Total** | | | **4,211,146** |

With stride 1 and no padding, each 7×7 convolution shrinks H and W by 6
(`O = I − K + 1`). Each kernel is as deep as its input, so one kernel gives one feature map.

In [8]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,7),
            nn.ReLU(),
            nn.Conv2d(32,64,7),
            nn.ReLU(),
            nn.Conv2d(64,128,7),
            nn.ReLU(),
            nn.Conv2d(128,256,7),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16384, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
        
        # He init cho cả Conv2d lẫn Linear
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Sanity check: kernel shape, output shape, parameter count

In [9]:
images, labels = next(iter(train_loader))   # lấy 1 batch để kiểm tra

model = CNN()
print(model.features[0].weight.shape)
print(model(images).shape)
summary(CNN(), (3, 32, 32), device="cpu")

torch.Size([32, 3, 7, 7])
torch.Size([64, 10])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 26, 26]           4,736
              ReLU-2           [-1, 32, 26, 26]               0
            Conv2d-3           [-1, 64, 20, 20]         100,416
              ReLU-4           [-1, 64, 20, 20]               0
            Conv2d-5          [-1, 128, 14, 14]         401,536
              ReLU-6          [-1, 128, 14, 14]               0
            Conv2d-7            [-1, 256, 8, 8]       1,605,888
              ReLU-8            [-1, 256, 8, 8]               0
           Flatten-9                [-1, 16384]               0
           Linear-10                  [-1, 128]       2,097,280
             ReLU-11                  [-1, 128]               0
           Linear-12                   [-1, 10]           1,290
Total params: 4,211,146
Trainable params: 4,211,146
Non-

## 3. Training setup

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

torch.manual_seed(42)
model = CNN(10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

device: mps


In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)  # forward
        loss = criterion(logits, labels) # compute loss

        optimizer.zero_grad()   # clear gradient
        loss.backward()         # backprop
        optimizer.step()        # update params

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

def evaluate(model, loader, criterion, device):
    model.eval()                                   # chế độ đánh giá
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():                          # không tính gradient
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n

## 4. Training (10 epochs)

In [12]:
epochs = 10
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"test loss {te_loss:.4f} acc {te_acc:.4f}")

Epoch  1 | train loss 1.6562 acc 0.3959 | test loss 1.4631 acc 0.4809
Epoch  2 | train loss 1.3175 acc 0.5280 | test loss 1.2462 acc 0.5556
Epoch  3 | train loss 1.1453 acc 0.5964 | test loss 1.1789 acc 0.5847
Epoch  4 | train loss 1.0073 acc 0.6462 | test loss 1.1094 acc 0.6098
Epoch  5 | train loss 0.8856 acc 0.6916 | test loss 1.0822 acc 0.6211
Epoch  6 | train loss 0.7699 acc 0.7333 | test loss 1.0843 acc 0.6317
Epoch  7 | train loss 0.6591 acc 0.7699 | test loss 1.0656 acc 0.6473
Epoch  8 | train loss 0.5533 acc 0.8095 | test loss 1.1064 acc 0.6484
Epoch  9 | train loss 0.4457 acc 0.8467 | test loss 1.1409 acc 0.6430
Epoch 10 | train loss 0.3558 acc 0.8787 | test loss 1.2556 acc 0.6360


## Results

Setup: Adam (lr = 1e-4), He (Kaiming uniform) initialisation, ReLU, batch size 64,
10 epochs, `torch.manual_seed(42)`. Accuracy reported at the final epoch.

| Model | Params | Train acc | Test acc |
|---|---|---|---|
| MLP, 1 hidden layer (Case 2) | 789,258 | 50.0% | 48.1% |
| CNN, 4 × Conv 7×7 | 4,211,146 | 87.9% | 63.6% |

### Observations
- With the same training budget, the CNN beats the MLP by about 15.5 points of test
  accuracy (63.6% vs 48.1%).
- The CNN overfits clearly. Test loss is lowest at epoch 7 (1.066) and rises to 1.256 by
  epoch 10, while train loss keeps falling (0.66 → 0.36). The train–test gap reaches
  about 24 points. The MLP never overfit within 10 epochs: it lacked the capacity to.
- About half the parameters (2.1M of 4.2M) sit in the first `Linear` layer, because the
  feature map is still 256 × 8 × 8 when flattened. Shrinking it first would cut this.
- Next steps from the lecture: down-sampling with max/average pooling, stride-2
  convolutions and "same" padding.